In [16]:
from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F



In [17]:
tokenizor=BertTokenizer.from_pretrained("bert-base-uncased")
model=BertForMaskedLM.from_pretrained("bert-base-uncased")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [18]:
text="The capital city of france is [MASK]."

In [19]:
inputs=tokenizor(text,return_tensors="pt")

In [20]:
with torch.inference_mode():
    logits=model(**input).logits

In [21]:
mask_index = (inputs.input_ids == tokenizor.mask_token_id).nonzero(as_tuple=True)[1][0].item()


In [22]:
mask_logits = logits[0, mask_index, :]

In [23]:
probs = F.softmax(mask_logits, dim=-1)
top_k = torch.topk(probs, k=5)

In [24]:
print("Top 5 predictions for [MASK]:")
for token_id, prob in zip(top_k.indices.tolist(), top_k.values.tolist()):
    token = tokenizor.convert_ids_to_tokens([token_id])[0]  
    print(f"{token:<12} -> {prob:.4f}")

Top 5 predictions for [MASK]:
.            -> 0.7253
;            -> 0.2424
|            -> 0.0284
?            -> 0.0017
!            -> 0.0017
